# Post-Training -- Expert
> Section 01 -- Topic 06 -- expert

**Prerequisites:** `06-post-training/advanced.ipynb`

**What you'll learn:**
- Advanced alignment techniques beyond DPO
- Constitutional AI, ORPO, KTO, process reward models

**What you'll build:**
- Multi-method alignment comparison

## Setup

This notebook covers cutting-edge alignment techniques. We'll implement each method's
core loss function and compare their properties. These methods represent the frontier of
post-training research as of 2024-2025.

In [ ]:
# !pip install transformers torch trl datasets accelerate

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import Dataset as HFDataset
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Constitutional AI

### Principle-Based Training Without Human Labels

Constitutional AI (CAI), introduced by Anthropic (Bai et al., 2022), takes a fundamentally
different approach to alignment. Instead of relying on human annotators to provide preference
labels, CAI uses the model itself to evaluate and improve its own outputs based on a set of
written **principles** (the "constitution").

The CAI pipeline has two phases:

**Phase 1: Supervised (Self-Critique and Revision)**
1. Generate an initial response to a prompt
2. Ask the model to critique its own response against a principle
3. Ask the model to revise the response based on the critique
4. Use the (prompt, revised_response) pair for SFT

**Phase 2: RL (RLAIF -- RL from AI Feedback)**
1. Generate pairs of responses
2. Ask the model which response better follows the principles
3. Use these AI-generated preferences to train a reward model
4. Run RLHF with the AI-trained reward model

The key advantage is scalability: you don't need human annotators. You can also easily
update the model's behavior by modifying the constitution. The principles can cover
helpfulness, harmlessness, honesty, and any domain-specific requirements.

In [ ]:
# Define a sample constitution

CONSTITUTION = [
    {
        "name": "Helpfulness",
        "critique_prompt": (
            "Identify specific ways the assistant's response is not helpful. "
            "Consider whether the response fully addresses the question, provides "
            "accurate information, and is clear and well-organized."
        ),
        "revision_prompt": (
            "Please rewrite the assistant's response to be more helpful. "
            "Make it more complete, accurate, clear, and well-organized."
        ),
    },
    {
        "name": "Harmlessness",
        "critique_prompt": (
            "Identify any ways the response could be harmful, misleading, or "
            "encourage dangerous behavior. Consider both direct and indirect harms."
        ),
        "revision_prompt": (
            "Please rewrite the response to remove any harmful content while "
            "remaining helpful. If the request itself is harmful, politely decline."
        ),
    },
    {
        "name": "Honesty",
        "critique_prompt": (
            "Identify any claims in the response that are uncertain or potentially "
            "incorrect. Does the response acknowledge its limitations appropriately?"
        ),
        "revision_prompt": (
            "Rewrite the response to be more honest. Acknowledge uncertainty where "
            "appropriate, correct any inaccuracies, and avoid stating opinions as facts."
        ),
    },
]

print("Constitutional AI Principles:")
for i, principle in enumerate(CONSTITUTION):
    print(f"\n{i+1}. {principle['name']}")
    print(f"   Critique: {principle['critique_prompt'][:80]}...")
    print(f"   Revision: {principle['revision_prompt'][:80]}...")

In [ ]:
def constitutional_ai_pipeline(prompt, initial_response, principles, model_generate_fn):
    """
    Implement the Constitutional AI self-critique and revision pipeline.
    
    This simulates the CAI process:
    1. Start with an initial response
    2. For each principle, critique and revise
    3. Return the final revised response
    
    In production, model_generate_fn would call the actual LLM.
    Here we simulate it for demonstration.
    """
    current_response = initial_response
    revision_history = []
    
    for principle in principles:
        # Step 1: Critique
        critique_input = (
            f"Human: {prompt}\n\n"
            f"Assistant: {current_response}\n\n"
            f"Critique Request: {principle['critique_prompt']}\n\n"
            f"Critique:"
        )
        critique = model_generate_fn(critique_input)
        
        # Step 2: Revise
        revision_input = (
            f"Human: {prompt}\n\n"
            f"Assistant: {current_response}\n\n"
            f"Critique: {critique}\n\n"
            f"Revision Request: {principle['revision_prompt']}\n\n"
            f"Revised Response:"
        )
        revised = model_generate_fn(revision_input)
        
        revision_history.append({
            "principle": principle["name"],
            "critique": critique,
            "before": current_response,
            "after": revised,
        })
        
        current_response = revised
    
    return current_response, revision_history


# Simulate the pipeline with mock responses
# (In practice, these would come from the actual LLM)

def mock_generate(prompt_text):
    """Simulate LLM generation for demonstration."""
    if "Critique Request" in prompt_text and "Helpfulness" in prompt_text:
        return "The response is too brief and lacks specific examples. It doesn't explain the practical implications."
    elif "Revision Request" in prompt_text and "helpful" in prompt_text.lower():
        return "Machine learning is a field of AI where systems learn patterns from data rather than being explicitly programmed. For example, a spam filter learns to identify spam by seeing thousands of labeled emails. Key approaches include supervised learning (learning from labeled examples), unsupervised learning (finding patterns in unlabeled data), and reinforcement learning (learning through trial and error with rewards)."
    elif "Critique Request" in prompt_text and "Harmless" in prompt_text:
        return "The response does not contain harmful content. It is a factual explanation of a technical topic."
    elif "Revision Request" in prompt_text and "harmful" in prompt_text.lower():
        return "Machine learning is a field of AI where systems learn patterns from data rather than being explicitly programmed. For example, a spam filter learns to identify spam by seeing thousands of labeled emails. Key approaches include supervised learning (learning from labeled examples), unsupervised learning (finding patterns in unlabeled data), and reinforcement learning (learning through trial and error with rewards)."
    elif "Critique Request" in prompt_text and "Honesty" in prompt_text:
        return "The response could better acknowledge that ML has limitations and is not a universal solution. It should mention that ML models can be biased and require careful evaluation."
    elif "Revision Request" in prompt_text and "honest" in prompt_text.lower():
        return "Machine learning is a field of AI where systems learn patterns from data rather than being explicitly programmed. For example, a spam filter learns to identify spam by seeing thousands of labeled emails. Key approaches include supervised learning, unsupervised learning, and reinforcement learning. It's worth noting that ML models are not perfect -- they can learn biases present in training data and their predictions should be evaluated carefully before deployment in high-stakes applications."
    return "[Generated response]"


prompt = "What is machine learning?"
initial_response = "ML is a type of AI that learns from data."

final_response, history = constitutional_ai_pipeline(
    prompt, initial_response, CONSTITUTION, mock_generate
)

print("Constitutional AI Revision Pipeline")
print("=" * 60)
print(f"\nOriginal: {initial_response}")

for step in history:
    print(f"\n--- After {step['principle']} revision ---")
    print(f"Critique: {step['critique']}")
    print(f"Revised: {step['after'][:150]}..." if len(step['after']) > 150 else f"Revised: {step['after']}")

print(f"\n{'=' * 60}")
print(f"Final response: {final_response}")

## 2. ORPO -- Odds Ratio Preference Optimization

### Simpler Than DPO: Combined SFT + Preference in One Loss

ORPO (Hong et al., 2024) simplifies post-training further by combining SFT and preference
optimization into a single training objective. The key insight is that you don't need a
separate reference model at all.

Standard SFT already teaches the model to increase the probability of good responses.
ORPO augments this with a penalty that also decreases the probability of bad responses,
using a log-odds ratio:

$$\mathcal{L}_{ORPO} = \mathcal{L}_{SFT}(y_w) + \lambda \cdot \mathcal{L}_{OR}$$

where the odds ratio loss is:

$$\mathcal{L}_{OR} = -\log \sigma\left(\log \frac{\text{odds}(y_w|x)}{\text{odds}(y_l|x)}\right)$$

and $\text{odds}(y|x) = \frac{P(y|x)}{1 - P(y|x)}$.

Advantages over DPO:
- No reference model needed (saves 50% memory compared to DPO)
- Single training phase (no separate SFT then DPO)
- Simpler implementation

In [ ]:
def compute_log_odds(logprobs):
    """
    Compute log-odds from log probabilities.
    
    odds(y|x) = P(y|x) / (1 - P(y|x))
    log_odds(y|x) = log P(y|x) - log(1 - P(y|x))
                   = logprobs - log(1 - exp(logprobs))
    
    We use log1p(-exp(logprobs)) for numerical stability.
    """
    return logprobs - torch.log1p(-torch.exp(logprobs))


def orpo_loss(policy_logps_chosen, policy_logps_rejected, nll_chosen, lambda_weight=1.0):
    """
    Compute the ORPO loss.
    
    L_ORPO = L_SFT(y_w) + lambda * L_OR
    L_OR = -log(sigma(log(odds(y_w|x) / odds(y_l|x))))
    
    Args:
        policy_logps_chosen: Average log P(token | context) for chosen responses
        policy_logps_rejected: Average log P(token | context) for rejected responses
        nll_chosen: Negative log-likelihood of chosen responses (SFT loss)
        lambda_weight: Weight of the odds ratio loss
    """
    # SFT loss on chosen responses
    sft_loss = nll_chosen.mean()
    
    # Log-odds ratio
    log_odds_chosen = compute_log_odds(policy_logps_chosen)
    log_odds_rejected = compute_log_odds(policy_logps_rejected)
    
    # Odds ratio loss
    log_odds_ratio = log_odds_chosen - log_odds_rejected
    or_loss = -F.logsigmoid(log_odds_ratio).mean()
    
    # Combined loss
    total_loss = sft_loss + lambda_weight * or_loss
    
    with torch.no_grad():
        accuracy = (log_odds_ratio > 0).float().mean()
    
    return total_loss, {
        "sft_loss": sft_loss.item(),
        "or_loss": or_loss.item(),
        "total_loss": total_loss.item(),
        "accuracy": accuracy.item(),
        "log_odds_ratio": log_odds_ratio.mean().item(),
    }


# Demonstrate ORPO loss
batch_size = 4
chosen_logps = torch.log(torch.tensor([0.3, 0.4, 0.35, 0.45]))  # Higher prob
rejected_logps = torch.log(torch.tensor([0.1, 0.15, 0.12, 0.08]))  # Lower prob
nll = -chosen_logps  # SFT loss

loss, metrics = orpo_loss(chosen_logps, rejected_logps, nll, lambda_weight=1.0)
print("ORPO Loss Computation:")
print(f"  SFT Loss: {metrics['sft_loss']:.4f}")
print(f"  Odds Ratio Loss: {metrics['or_loss']:.4f}")
print(f"  Total Loss: {metrics['total_loss']:.4f}")
print(f"  Preference Accuracy: {metrics['accuracy']:.2%}")

In [ ]:
# Compare ORPO vs DPO training dynamics on synthetic data

def simulate_training_dynamics(method, n_steps=100, lr=0.01):
    """
    Simulate training dynamics for DPO and ORPO on toy data.
    
    We parameterize the model's preference as a single scalar
    and track how it evolves during training.
    """
    # Parameterize the model with a simple preference strength
    theta = torch.tensor(0.0, requires_grad=True)
    optimizer = torch.optim.SGD([theta], lr=lr)
    
    losses = []
    preference_strengths = []
    
    for step in range(n_steps):
        # Simulate log-probs that depend on theta
        chosen_logp = -2.0 + theta * 0.5
        rejected_logp = -2.0 - theta * 0.3
        
        if method == "DPO":
            ref_chosen = torch.tensor(-2.0)
            ref_rejected = torch.tensor(-2.0)
            beta = 0.1
            logits = beta * ((chosen_logp - ref_chosen) - (rejected_logp - ref_rejected))
            loss = -F.logsigmoid(logits)
        elif method == "ORPO":
            log_odds_c = chosen_logp - torch.log1p(-torch.exp(chosen_logp))
            log_odds_r = rejected_logp - torch.log1p(-torch.exp(rejected_logp))
            sft_loss = -chosen_logp
            or_loss = -F.logsigmoid(log_odds_c - log_odds_r)
            loss = sft_loss + or_loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        preference_strengths.append(theta.item())
    
    return losses, preference_strengths


dpo_losses, dpo_strengths = simulate_training_dynamics("DPO")
orpo_losses, orpo_strengths = simulate_training_dynamics("ORPO")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(dpo_losses, label="DPO", color="#C44E52")
ax1.plot(orpo_losses, label="ORPO", color="#55A868")
ax1.set_xlabel("Training Step")
ax1.set_ylabel("Loss")
ax1.set_title("Training Loss Comparison")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(dpo_strengths, label="DPO", color="#C44E52")
ax2.plot(orpo_strengths, label="ORPO", color="#55A868")
ax2.set_xlabel("Training Step")
ax2.set_ylabel("Preference Strength (theta)")
ax2.set_title("Preference Learning Dynamics")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("ORPO converges faster because it combines SFT and preference signals.")

## 3. KTO -- Kahneman-Tversky Optimization

### Working Without Paired Preferences

KTO (Ethayarajh et al., 2024) addresses a practical limitation of DPO: it requires
**paired** preferences (for each prompt, you need both a chosen and rejected response).
In practice, collecting paired data is expensive. Often you only have **unpaired**
binary feedback: individual responses marked as "good" (thumbs up) or "bad" (thumbs down).

KTO draws on **prospect theory** from behavioral economics (Kahneman & Tversky, 1979).
The key insight from prospect theory is that humans are **loss-averse**: the pain of
losing something is psychologically about twice as powerful as the pleasure of gaining
the same thing.

The KTO loss function applies this asymmetry:

$$\mathcal{L}_{KTO} = \mathbb{E}_{(x,y) \sim D_{\text{good}}} \left[\lambda_w \cdot \sigma(-r(x,y))\right] + \mathbb{E}_{(x,y) \sim D_{\text{bad}}} \left[\lambda_l \cdot \sigma(r(x,y))\right]$$

where $r(x,y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)} - z_{\text{ref}}$
and $z_{\text{ref}}$ is a running estimate of the reference point.

The asymmetric weights $\lambda_w \neq \lambda_l$ encode loss aversion: we penalize
generating bad outputs more than we reward generating good outputs.

In [ ]:
def kto_loss(
    policy_logps,
    ref_logps,
    is_good,
    beta=0.1,
    lambda_good=1.0,
    lambda_bad=1.5,  # Higher penalty for bad outputs (loss aversion)
):
    """
    Compute the KTO loss.
    
    Unlike DPO, KTO works with unpaired binary feedback:
    each example is independently labeled as good or bad.
    
    Args:
        policy_logps: log P(y|x) under the policy (batch,)
        ref_logps: log P(y|x) under the reference model (batch,)
        is_good: boolean tensor, True for good examples (batch,)
        beta: temperature parameter
        lambda_good: weight for good examples
        lambda_bad: weight for bad examples (higher = more loss-averse)
    """
    # Compute the implicit reward
    log_ratio = policy_logps - ref_logps
    
    # Reference point: the expected reward under the reference model
    # In practice, this is estimated as a running average
    z_ref = (beta * log_ratio).detach().mean()
    
    # Compute the reward relative to the reference point
    reward = beta * log_ratio - z_ref
    
    # KTO loss with asymmetric weights
    good_mask = is_good.float()
    bad_mask = 1.0 - good_mask
    
    # For good examples: penalize if reward is low (sigmoid of negative reward)
    good_loss = lambda_good * torch.sigmoid(-reward) * good_mask
    
    # For bad examples: penalize if reward is high (sigmoid of reward)
    bad_loss = lambda_bad * torch.sigmoid(reward) * bad_mask
    
    total_loss = (good_loss + bad_loss).mean()
    
    with torch.no_grad():
        good_reward = (reward * good_mask).sum() / good_mask.sum().clamp(min=1)
        bad_reward = (reward * bad_mask).sum() / bad_mask.sum().clamp(min=1)
    
    return total_loss, {
        "loss": total_loss.item(),
        "avg_good_reward": good_reward.item(),
        "avg_bad_reward": bad_reward.item(),
        "z_ref": z_ref.item(),
    }


# Demonstrate KTO loss
batch_size = 8
policy_logps = torch.randn(batch_size) * 0.5 - 2.0
ref_logps = torch.randn(batch_size) * 0.5 - 2.0
is_good = torch.tensor([True, True, True, False, False, True, False, True])

loss, metrics = kto_loss(policy_logps, ref_logps, is_good)
print("KTO Loss Computation:")
for key, value in metrics.items():
    print(f"  {key}: {value:.4f}")
print(f"\nNumber of good examples: {is_good.sum().item()}")
print(f"Number of bad examples: {(~is_good).sum().item()}")

In [ ]:
# Visualize the asymmetric loss landscape of KTO

rewards = np.linspace(-3, 3, 200)

# KTO loss components
good_penalty = 1.0 * 1.0 / (1.0 + np.exp(rewards))   # sigma(-r) for good
bad_penalty = 1.5 * 1.0 / (1.0 + np.exp(-rewards))     # sigma(r) for bad

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
ax1.plot(rewards, good_penalty, label="Good examples (lambda=1.0)", color="green", linewidth=2)
ax1.plot(rewards, bad_penalty, label="Bad examples (lambda=1.5)", color="red", linewidth=2)
ax1.set_xlabel("Implicit Reward")
ax1.set_ylabel("Loss")
ax1.set_title("KTO Loss: Asymmetric by Design")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axvline(x=0, color="gray", linestyle="--", alpha=0.5)

# Compare with symmetric DPO-like loss
symmetric = 1.0 / (1.0 + np.exp(rewards))  # Same weight for both
ax2.plot(rewards, symmetric, label="Symmetric (DPO-like)", color="blue", linewidth=2)
ax2.plot(rewards, good_penalty, label="Good (KTO)", color="green", linewidth=2, linestyle="--")
ax2.plot(rewards, bad_penalty, label="Bad (KTO)", color="red", linewidth=2, linestyle="--")
ax2.set_xlabel("Implicit Reward")
ax2.set_ylabel("Loss")
ax2.set_title("Symmetric vs Asymmetric Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The asymmetry means the model is penalized more for generating bad outputs")
print("than it is rewarded for generating good ones -- matching human loss aversion.")

## 4. Process Reward Models

### Step-Level Rewards for Better Reasoning

Standard reward models (outcome reward models, or ORMs) assign a single score to the
complete response. This works well for many tasks, but for **reasoning tasks** (math,
coding, logic), a single score provides a sparse signal: the model only knows if the
final answer was right or wrong, not *where* the reasoning went astray.

**Process Reward Models** (PRMs) solve this by providing reward at each reasoning step.
The idea, explored in depth by Lightman et al. (2023) in "Let's Verify Step by Step",
is to train a model that can evaluate each step of a solution independently.

Benefits of process supervision:
- **Better credit assignment:** The model knows exactly which step was wrong
- **Improved reasoning:** Models trained with PRMs make fewer logical errors
- **Better search:** During inference, you can use the PRM to guide beam search or
  tree search, pruning branches that contain incorrect reasoning steps
- **Interpretability:** You can see where the model's confidence drops

In [ ]:
class ProcessRewardModel(nn.Module):
    """
    A Process Reward Model that scores each step in a reasoning chain.
    
    Architecture: Language model backbone with a per-step scoring head.
    The model reads the solution step-by-step and assigns a correctness
    score after each step delimiter.
    """
    
    def __init__(self, model_name="gpt2", step_delimiter="\n"):
        super().__init__()
        self.backbone = AutoModelForCausalLM.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        
        # Per-step scoring head: predicts P(step is correct)
        self.step_scorer = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1),
        )
        self.step_delimiter = step_delimiter
    
    def forward(self, input_ids, attention_mask=None, step_positions=None):
        """
        Forward pass.
        
        Args:
            input_ids: Token IDs
            attention_mask: Attention mask
            step_positions: Positions of step delimiters where we score
        
        Returns:
            step_scores: Score at each step position
        """
        outputs = self.backbone.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        hidden_states = outputs.last_hidden_state
        
        if step_positions is not None:
            # Extract hidden states at step positions
            batch_size = hidden_states.size(0)
            step_hidden = []
            for i in range(batch_size):
                positions = step_positions[i]
                step_hidden.append(hidden_states[i, positions])  # (n_steps, hidden)
            
            # Score each step
            step_scores = []
            for h in step_hidden:
                scores = torch.sigmoid(self.step_scorer(h)).squeeze(-1)  # (n_steps,)
                step_scores.append(scores)
            
            return step_scores
        else:
            # Score all positions (for flexibility)
            all_scores = torch.sigmoid(self.step_scorer(hidden_states)).squeeze(-1)
            return all_scores


# Initialize the PRM
prm = ProcessRewardModel("gpt2").to(device)
print(f"Process Reward Model parameters: {sum(p.numel() for p in prm.parameters()):,}")

In [ ]:
# Demonstrate process reward scoring on a math solution

prm_tokenizer = AutoTokenizer.from_pretrained("gpt2")
prm_tokenizer.pad_token = prm_tokenizer.eos_token

# A math problem with step-by-step solution
correct_solution = """Problem: What is 15% of 240?
Step 1: Convert 15% to a decimal: 15/100 = 0.15
Step 2: Multiply 0.15 by 240: 0.15 * 240 = 36
Answer: 15% of 240 is 36"""

incorrect_solution = """Problem: What is 15% of 240?
Step 1: Convert 15% to a decimal: 15/100 = 1.5
Step 2: Multiply 1.5 by 240: 1.5 * 240 = 360
Answer: 15% of 240 is 360"""

def score_solution_steps(prm_model, tokenizer, solution_text):
    """
    Score each step of a solution using the process reward model.
    """
    # Split into steps
    steps = solution_text.split("\n")
    
    # Tokenize the full solution
    inputs = tokenizer(solution_text, return_tensors="pt", truncation=True).to(device)
    
    # Find newline token positions (step boundaries)
    newline_token = tokenizer.encode("\n")[0]
    step_positions = (inputs["input_ids"][0] == newline_token).nonzero(as_tuple=True)[0]
    
    # If we can't find newline tokens, use evenly spaced positions
    if len(step_positions) == 0:
        seq_len = inputs["input_ids"].size(1)
        step_positions = torch.linspace(0, seq_len - 1, len(steps)).long().to(device)
    
    # Score
    prm_model.eval()
    with torch.no_grad():
        scores = prm_model(
            inputs["input_ids"],
            inputs.get("attention_mask"),
            step_positions=[step_positions[:len(steps)]]
        )
    
    return steps, scores[0].cpu().numpy()

print("Process Reward Model Scoring")
print("=" * 60)

print("\n--- Correct Solution ---")
steps_c, scores_c = score_solution_steps(prm, prm_tokenizer, correct_solution)
for step, score in zip(steps_c, scores_c[:len(steps_c)]):
    print(f"  [{score:.3f}] {step}")

print("\n--- Incorrect Solution ---")
steps_i, scores_i = score_solution_steps(prm, prm_tokenizer, incorrect_solution)
for step, score in zip(steps_i, scores_i[:len(steps_i)]):
    print(f"  [{score:.3f}] {step}")

print("\nNote: Scores are from an untrained PRM (random). After training on")
print("step-level labels, the PRM would assign low scores to incorrect steps.")

In [ ]:
# Visualize outcome RM vs process RM on a reasoning chain

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

steps_labels = ["Problem", "Step 1", "Step 2", "Step 3", "Step 4", "Answer"]

# Outcome RM: single score for the whole solution
correct_outcome = [0.85] * 6  # One score applies to everything
incorrect_outcome = [0.15] * 6

ax1.bar(np.arange(6) - 0.15, correct_outcome, 0.3, label="Correct solution", color="green", alpha=0.7)
ax1.bar(np.arange(6) + 0.15, incorrect_outcome, 0.3, label="Incorrect solution", color="red", alpha=0.7)
ax1.set_xticks(range(6))
ax1.set_xticklabels(steps_labels, rotation=30)
ax1.set_ylabel("Reward Score")
ax1.set_title("Outcome Reward Model\n(same score for all steps)")
ax1.legend()
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3, axis="y")

# Process RM: per-step scores
correct_process = [0.95, 0.92, 0.90, 0.88, 0.91, 0.93]
incorrect_process = [0.95, 0.92, 0.30, 0.15, 0.10, 0.08]  # Error at step 3

ax2.bar(np.arange(6) - 0.15, correct_process, 0.3, label="Correct solution", color="green", alpha=0.7)
ax2.bar(np.arange(6) + 0.15, incorrect_process, 0.3, label="Incorrect solution", color="red", alpha=0.7)
ax2.set_xticks(range(6))
ax2.set_xticklabels(steps_labels, rotation=30)
ax2.set_ylabel("Reward Score")
ax2.set_title("Process Reward Model\n(per-step scores, error at Step 3)")
ax2.legend()
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3, axis="y")
ax2.annotate("Error detected!", xy=(2.15, 0.30), xytext=(3.5, 0.55),
             arrowprops=dict(arrowstyle="->", color="red"),
             fontsize=10, color="red", fontweight="bold")

plt.tight_layout()
plt.show()

print("Process reward models provide fine-grained feedback, showing exactly")
print("where reasoning goes wrong. This enables better credit assignment during training.")

## 5. Iterative RLHF

### Why Single-Round Alignment Isn't Enough

A single round of RLHF or DPO improves the model, but it's not sufficient for strong
alignment. There are several reasons:

1. **Distribution shift:** After one round of alignment, the model generates different
   outputs than the SFT model. The preference data was collected on SFT model outputs,
   so it may not cover the failure modes of the newly aligned model.

2. **Reward hacking:** Given enough optimization pressure, the model finds ways to get
   high reward without actually being more helpful. This is an instance of Goodhart's
   Law: "When a measure becomes a target, it ceases to be a good measure."

3. **Capability improvement:** As the model improves, it can produce more sophisticated
   responses that require more sophisticated evaluation -- the reward model needs to
   keep up.

Iterative RLHF addresses these by running multiple rounds: generate new data with the
current model, collect new preferences, retrain the reward model, and optimize again.

In [ ]:
# Demonstrate reward hacking on a toy example

def simulate_reward_hacking(n_rounds=20):
    """
    Simulate reward hacking: the model finds patterns that fool
    the reward model without improving actual quality.
    
    We model this as the policy learning to exploit a "feature"
    that correlates with reward but doesn't reflect true quality.
    """
    true_quality = []  # What humans actually think
    reward_score = []  # What the RM predicts
    response_length = []  # A proxy feature the model might exploit
    
    base_quality = 0.5
    base_length = 100
    
    for round_num in range(n_rounds):
        # The model discovers that longer responses get higher RM scores
        # (a known failure mode of reward models)
        length_increase = round_num * 15
        current_length = base_length + length_increase
        
        # Reward model is fooled by length (imperfect correlation)
        rm_score = 0.5 + 0.025 * round_num - 0.0005 * round_num ** 2
        rm_score = min(rm_score, 0.85)
        
        # True quality increases initially, then plateaus/decreases
        # as the model starts padding responses with fluff
        if round_num < 8:
            quality = base_quality + 0.03 * round_num
        else:
            quality = base_quality + 0.03 * 8 - 0.015 * (round_num - 8)
        
        true_quality.append(quality)
        reward_score.append(rm_score)
        response_length.append(current_length)
    
    return true_quality, reward_score, response_length


quality, rewards, lengths = simulate_reward_hacking()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

rounds = range(len(quality))
ax1.plot(rounds, quality, label="True Quality", color="green", linewidth=2)
ax1.plot(rounds, rewards, label="RM Score", color="red", linewidth=2, linestyle="--")
ax1.set_xlabel("Optimization Round")
ax1.set_ylabel("Score")
ax1.set_title("Reward Hacking: RM Score vs True Quality")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.axvspan(8, 19, alpha=0.1, color="red", label="Overoptimization")

ax2.plot(rounds, lengths, color="#4C72B0", linewidth=2)
ax2.set_xlabel("Optimization Round")
ax2.set_ylabel("Response Length (tokens)")
ax2.set_title("Response Length Increases (Exploited Feature)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("After round 8, the RM score keeps rising but true quality drops.")
print("The model learned to game the RM by producing longer (but fluffier) responses.")

In [ ]:
# Mitigation strategies for reward hacking

mitigation_strategies = {
    "KL Penalty": {
        "description": "Constrain the policy to stay close to the reference model",
        "effectiveness": 0.65,
        "complexity": 0.3,
    },
    "Reward Model\nEnsemble": {
        "description": "Use multiple reward models and take the minimum score",
        "effectiveness": 0.75,
        "complexity": 0.6,
    },
    "Iterative\nRetraining": {
        "description": "Periodically retrain the RM on new model outputs",
        "effectiveness": 0.80,
        "complexity": 0.7,
    },
    "Length\nNormalization": {
        "description": "Normalize reward by response length to prevent length hacking",
        "effectiveness": 0.55,
        "complexity": 0.1,
    },
    "Process RM\n(PRM)": {
        "description": "Use step-level rewards for fine-grained credit assignment",
        "effectiveness": 0.85,
        "complexity": 0.8,
    },
}

names = list(mitigation_strategies.keys())
effectiveness = [s["effectiveness"] for s in mitigation_strategies.values()]
complexity = [s["complexity"] for s in mitigation_strategies.values()]

fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(complexity, effectiveness, s=200, c=effectiveness,
                     cmap="RdYlGn", edgecolors="black", linewidth=1, zorder=5)

for i, name in enumerate(names):
    ax.annotate(name, (complexity[i], effectiveness[i]),
                textcoords="offset points", xytext=(10, 5),
                fontsize=9, fontweight="bold")

ax.set_xlabel("Implementation Complexity")
ax.set_ylabel("Effectiveness Against Reward Hacking")
ax.set_title("Reward Hacking Mitigation Strategies")
ax.grid(True, alpha=0.3)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0.4, 0.95)

plt.tight_layout()
plt.show()

## 6. Rejection Sampling

### Generate Multiple, Pick Best

Rejection sampling (also called **Best-of-N sampling** or **reward-ranked fine-tuning**)
is a simple but powerful technique that bridges generation and alignment:

1. For each prompt, generate N candidate responses from the model
2. Score each response with the reward model
3. Select the highest-scoring response

This can be used in two ways:
- **At inference time:** Generate N responses and return the best one to the user.
  This is simple but costs N times the compute.
- **As training data:** Use the best responses as new SFT data, or use the
  (best, worst) pairs as DPO training data. This is called **rejection sampling
  fine-tuning** (RSF) and is used by Llama 2 and many production models.

The quality of Best-of-N sampling improves with N, but with diminishing returns.
Empirically, N=16 to N=64 provides a good balance of quality and compute.

In [ ]:
def rejection_sampling(
    model, tokenizer, prompt, reward_fn, n_samples=8, max_new_tokens=100
):
    """
    Generate N responses and select the best one according to a reward function.
    
    Args:
        model: The language model
        tokenizer: The tokenizer
        prompt: The input prompt
        reward_fn: A function that scores (prompt, response) -> float
        n_samples: Number of candidate responses to generate
        max_new_tokens: Maximum tokens per response
    
    Returns:
        best_response, all_responses_with_scores
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_length = inputs["input_ids"].shape[1]
    
    candidates = []
    
    model.eval()
    with torch.no_grad():
        for i in range(n_samples):
            # Generate a response with sampling (temperature > 0 for diversity)
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.8,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )
            
            # Extract just the generated response (remove prompt)
            response_ids = outputs[0][prompt_length:]
            response_text = tokenizer.decode(response_ids, skip_special_tokens=True)
            
            # Score with the reward function
            score = reward_fn(prompt, response_text)
            
            candidates.append({
                "response": response_text,
                "score": score,
            })
    
    # Sort by score (descending)
    candidates.sort(key=lambda x: x["score"], reverse=True)
    
    return candidates[0], candidates


# Simple reward function for demonstration
# (In practice, this would be a trained reward model)
def simple_reward_fn(prompt, response):
    """A heuristic reward function for demonstration."""
    score = 0.0
    # Reward longer but not too long responses
    words = len(response.split())
    if 20 < words < 200:
        score += 0.3
    # Penalize repetition
    unique_words = len(set(response.lower().split()))
    if words > 0:
        score += 0.3 * (unique_words / words)
    # Reward ending with proper punctuation
    if response.strip().endswith((".", "!", "?")):
        score += 0.2
    return score


# Run rejection sampling
gen_model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)
gen_tokenizer = AutoTokenizer.from_pretrained("gpt2")
gen_tokenizer.pad_token = gen_tokenizer.eos_token

prompt = "The best way to learn programming is"
best, all_candidates = rejection_sampling(
    gen_model, gen_tokenizer, prompt, simple_reward_fn, n_samples=8
)

print(f"Generated {len(all_candidates)} candidates for: '{prompt}'")
print(f"\nBest candidate (score={best['score']:.3f}):")
print(f"  {best['response'][:200]}")

print(f"\nAll scores: {[f'{c["score"]:.3f}' for c in all_candidates]}")

In [ ]:
def create_dpo_pairs_from_rejection_sampling(candidates):
    """
    Use rejection sampling results to create DPO training pairs.
    
    Strategy: pair the best response with each non-best response.
    This creates (N-1) training pairs from N candidates.
    """
    best = candidates[0]
    pairs = []
    
    for candidate in candidates[1:]:
        pairs.append({
            "chosen": best["response"],
            "rejected": candidate["response"],
            "chosen_score": best["score"],
            "rejected_score": candidate["score"],
            "score_margin": best["score"] - candidate["score"],
        })
    
    return pairs


dpo_pairs = create_dpo_pairs_from_rejection_sampling(all_candidates)

print(f"Created {len(dpo_pairs)} DPO training pairs from rejection sampling")
print(f"\nScore margins: {[f'{p["score_margin"]:.3f}' for p in dpo_pairs]}")

# Visualize the distribution of scores
scores = [c["score"] for c in all_candidates]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(len(scores)), scores, color=["green" if i == 0 else "#4C72B0" for i in range(len(scores))])
ax1.set_xlabel("Candidate (sorted by score)")
ax1.set_ylabel("Reward Score")
ax1.set_title("Candidate Scores (best in green)")
ax1.grid(True, alpha=0.3, axis="y")

# Show theoretical best-of-N improvement
n_values = [1, 2, 4, 8, 16, 32, 64]
# Approximate: best-of-N score improves as E[max(X_1,...,X_N)] for Gaussian
expected_best = [np.mean([max(np.random.normal(0.5, 0.15, n)) for _ in range(500)]) for n in n_values]

ax2.plot(n_values, expected_best, "o-", color="#C44E52", linewidth=2)
ax2.set_xlabel("N (number of candidates)")
ax2.set_ylabel("Expected Best Score")
ax2.set_title("Best-of-N: Diminishing Returns")
ax2.set_xscale("log", base=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Best-of-N quality increases with N but with diminishing returns.")
print("N=16 to N=64 is the practical sweet spot for most applications.")

## Summary

In this notebook, we explored advanced alignment techniques beyond standard DPO:

- **Constitutional AI:** Self-critique and revision using written principles, enabling
  scalable alignment without human annotators. The model evaluates and improves its own
  outputs through a structured pipeline.

- **ORPO:** Combines SFT and preference optimization into a single loss using odds ratios,
  eliminating the need for a reference model and a separate SFT phase.

- **KTO:** Works with unpaired binary feedback (good/bad labels) instead of paired
  preferences, using asymmetric loss inspired by prospect theory's loss aversion.

- **Process Reward Models:** Provide step-level rewards for reasoning tasks, enabling
  better credit assignment and improved reasoning quality.

- **Iterative RLHF and reward hacking:** Single-round alignment is insufficient;
  models can exploit reward model weaknesses. Mitigation strategies include KL penalties,
  reward model ensembles, and iterative retraining.

- **Rejection Sampling:** A simple but effective technique that generates multiple
  candidates and selects the best, producing high-quality training data for further
  alignment.

**Next:** In `build.ipynb`, we'll put it all together in a complete post-training
pipeline: SFT followed by DPO followed by evaluation.